# What this notebook is doing

This notebook checks the native time resolution of the precipitation datasets used in the AR 6-hour work.

It goes through the available sources and figures out which ones are hourly, which ones are daily, and whether any have other time steps. It also records the time coordinate, date range, and precipitation variable for each dataset so it is easier to compare sources before any downstream processing.

At the end it prints the full audit table, shows a short summary by source and resolution, and saves the results to `BASE/derived/dataset_time_resolution_audit_all_sources.csv`.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import xarray as xr
import intake
from urllib.parse import urlsplit, urlunsplit

# ----------------------------
# PATHS / CONSTANTS
# ----------------------------
BASE = Path("/data0/balaji24/data")

P_PRISM_DIR = BASE / "prism_ppt"
P_HRRR_DIR  = BASE / "weather_data"

UCLA_PREC_DIR      = BASE / "ucla_era5_d02_daily" / "prec"
UCLA_PREC_TEMPLATE = "prec.daily.era5.d02.{year}.nc"

P_PNNL_DIR = BASE / "PNNL" / "historical"

CONUS_INTAKE_YML = (
    "https://raw.githubusercontent.com/hytest-org/hytest/main/"
    "dataset_catalog/hytest_intake_catalog.yml"
)

# ----------------------------
# GENERIC HELPERS
# ----------------------------
def safe_open_zarr(p: Path):
    try:
        return xr.open_zarr(p, consolidated=True)
    except Exception:
        return xr.open_zarr(p, consolidated=False)

def pick_time_coord(ds):
    preferred = ["time", "Time", "valid_time", "forecast_time"]
    for name in preferred:
        if name in ds.coords or name in ds.variables:
            return name
    for v in ds.variables:
        if "time" in v.lower():
            return v
    return None

def pick_main_var(ds):
    preferred = [
        "ppt", "precip", "prcp", "tp", "apcp",
        "prec", "PREC_ACC_NC", "RAINNC", "RAINC",
        "PREC", "PRECIP", "RAIN"
    ]
    for v in preferred:
        if v in ds.data_vars:
            return v
    if len(ds.data_vars) > 0:
        return list(ds.data_vars)[0]
    return None

def median_dt_hours(time_values):
    try:
        t = pd.to_datetime(time_values)
        if len(t) < 2:
            return np.nan
        dt = pd.Series(t).diff().dt.total_seconds().dropna()
        if len(dt) == 0:
            return np.nan
        return float(dt.median() / 3600.0)
    except Exception:
        return np.nan

def infer_resolution(hours):
    if pd.isna(hours):
        return "unknown"
    if abs(hours - 1) < 0.1:
        return "hourly"
    if abs(hours - 6) < 0.2:
        return "6-hourly"
    if abs(hours - 24) < 0.5:
        return "daily"
    if hours < 24:
        return f"sub-daily (~{hours:.2f}h)"
    return f">{hours:.2f}h"

def summarize_step_coord(ds):
    for name in ["step", "forecast_hour"]:
        if name in ds.coords or name in ds.variables:
            arr = ds[name].values
            try:
                if np.issubdtype(arr.dtype, np.timedelta64):
                    vals = np.unique(arr / np.timedelta64(1, "h"))
                    vals = np.asarray(vals, dtype=float)
                else:
                    vals = np.asarray(arr, dtype=float)
                vals = np.unique(vals)
                vals = vals[np.isfinite(vals)]
                if len(vals) == 0:
                    return f"{name}: present (empty)"
                if len(vals) <= 12:
                    return f"{name}: {vals.tolist()}"
                return f"{name}: min={vals.min():.2f}, max={vals.max():.2f}, count={len(vals)}"
            except Exception:
                return f"{name}: present"
    return "none"

def inspect_dataset(source, label, opener):
    ds = None
    try:
        ds = opener()
        tname = pick_time_coord(ds)
        vname = pick_main_var(ds)

        if tname is None:
            return {
                "source": source,
                "label": label,
                "var": vname,
                "time_coord": None,
                "n_time": None,
                "start": None,
                "end": None,
                "median_dt_hours": np.nan,
                "resolution": "no time coord",
                "forecast_meta": summarize_step_coord(ds),
            }

        t = ds[tname].values
        med = median_dt_hours(t)

        return {
            "source": source,
            "label": label,
            "var": vname,
            "time_coord": tname,
            "n_time": len(t),
            "start": str(pd.to_datetime(t[0])) if len(t) else None,
            "end": str(pd.to_datetime(t[-1])) if len(t) else None,
            "median_dt_hours": med,
            "resolution": infer_resolution(med),
            "forecast_meta": summarize_step_coord(ds),
        }
    except Exception as e:
        return {
            "source": source,
            "label": label,
            "var": None,
            "time_coord": None,
            "n_time": None,
            "start": None,
            "end": None,
            "median_dt_hours": np.nan,
            "resolution": f"open failed: {type(e).__name__}: {e}",
            "forecast_meta": None,
        }
    finally:
        if ds is not None:
            try:
                ds.close()
            except Exception:
                pass

# ----------------------------
# EXTERNAL: DAYMET
# ----------------------------
def ensure_daymet_pkgs():
    needed = ["pystac-client", "planetary-computer", "fsspec", "zarr"]
    import importlib, subprocess, sys
    missing = []
    for p in needed:
        mod = p.replace("-", "_")
        try:
            importlib.import_module(mod)
        except Exception:
            missing.append(p)
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

ensure_daymet_pkgs()

from pystac_client import Client
import planetary_computer as pc
from fsspec.implementations.http import HTTPFileSystem

class SASEndAppendingHTTPFileSystem(HTTPFileSystem):
    def __init__(self, sas_query: str, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._sas_query = sas_query.lstrip("?")

    def _with_sas(self, url: str) -> str:
        return url if "?" in url else f"{url}?{self._sas_query}"

    def _open(self, path, mode="rb", block_size=None, **kwargs):
        return super()._open(self._with_sas(path), mode=mode, block_size=block_size, **kwargs)

    async def _cat_file(self, url, start=None, end=None, **kwargs):
        return await super()._cat_file(self._with_sas(url), start=start, end=end, **kwargs)

def open_daymet_ds():
    stac = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")
    col = stac.get_collection("daymet-daily-na")
    asset_key = "zarr-https" if "zarr-https" in col.assets else "zarr-abfs"
    signed_href = pc.sign(col.assets[asset_key].href)

    parts = urlsplit(signed_href)
    root_no_query = urlunsplit((parts.scheme, parts.netloc, parts.path, "", ""))
    sas_query = parts.query

    fs = SASEndAppendingHTTPFileSystem(sas_query)
    mapper = fs.get_mapper(root_no_query)

    open_kwargs = dict(col.assets[asset_key].extra_fields.get("xarray:open_kwargs", {}))
    open_kwargs.pop("consolidated", None)

    ds = xr.open_zarr(mapper, **open_kwargs)
    return ds

# ----------------------------
# EXTERNAL: CONUS404
# ----------------------------
def open_conus_ds():
    cat = intake.open_catalog(CONUS_INTAKE_YML)
    ds = cat["conus404-catalog"]["conus404-daily-osn"].to_dask()
    return ds

# ----------------------------
# COLLECT INSPECTION ROWS
# ----------------------------
rows = []

# PRISM local zarrs
for p in sorted(P_PRISM_DIR.glob("*.zarr")):
    rows.append(
        inspect_dataset(
            source="PRISM",
            label=p.name,
            opener=lambda p=p: safe_open_zarr(p),
        )
    )

# HRRR local zarrs
for p in sorted(P_HRRR_DIR.glob("*.zarr")):
    rows.append(
        inspect_dataset(
            source="HRRR",
            label=p.name,
            opener=lambda p=p: safe_open_zarr(p),
        )
    )

# UCLA local nc
for p in sorted(UCLA_PREC_DIR.glob("*.nc")):
    rows.append(
        inspect_dataset(
            source="UCLA ERA5 d02",
            label=p.name,
            opener=lambda p=p: xr.open_dataset(p, engine="netcdf4"),
        )
    )

# PNNL local nc
for p in sorted(P_PNNL_DIR.glob("*/*PREC_ACC_NC*.nc")):
    rows.append(
        inspect_dataset(
            source="PNNL hist",
            label=str(p.relative_to(P_PNNL_DIR)),
            opener=lambda p=p: xr.open_dataset(p, engine="netcdf4"),
        )
    )

# Daymet external
rows.append(
    inspect_dataset(
        source="Daymet-PC",
        label="daymet-daily-na (Planetary Computer)",
        opener=open_daymet_ds,
    )
)

# CONUS404 external
rows.append(
    inspect_dataset(
        source="CONUS404",
        label="conus404-daily-osn (HyTEST intake)",
        opener=open_conus_ds,
    )
)

df = pd.DataFrame(rows)

# ----------------------------
# DISPLAY
# ----------------------------
cols = [
    "source", "label", "var", "time_coord", "n_time",
    "start", "end", "median_dt_hours", "resolution", "forecast_meta"
]

with pd.option_context("display.max_colwidth", 160, "display.max_rows", 500):
    print(df[cols].to_string(index=False))

print("\n=== SUMMARY BY SOURCE / RESOLUTION ===")
summary = (
    df.groupby(["source", "resolution"], dropna=False)
      .size()
      .reset_index(name="count")
      .sort_values(["source", "resolution"])
)
print(summary.to_string(index=False))

# Optional: save
OUT = BASE / "derived"
OUT.mkdir(parents=True, exist_ok=True)
out_csv = OUT / "dataset_time_resolution_audit_all_sources.csv"
df.to_csv(out_csv, index=False)
print(f"\nSaved audit CSV -> {out_csv}")

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7b49334d2030>
Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x7b49302e4e30>, 1301842.883528138), (<aiohttp.client_proto.ResponseHandler object at 0x7b493029d9d0>, 1301843.011357661), (<aiohttp.client_proto.ResponseHandler object at 0x7b493029daf0>, 1301843.022345965), (<aiohttp.client_proto.ResponseHandler object at 0x7b493029d010>, 1301843.024691968), (<aiohttp.client_proto.ResponseHandler object at 0x7b493029f5f0>, 1301843.024905761), (<aiohttp.client_proto.ResponseHandler object at 0x7b493029c410>, 1301843.034309793), (<aiohttp.client_proto.ResponseHandler object at 0x7b493029c050>, 1301843.036617296), (<aiohttp.client_proto.ResponseHandler object at 0x7b49302e4d70>, 1301843.038243979), (<aiohttp.client_proto.ResponseHandler object at 0x7b493029e9f0>, 1301843.038450692), (<aiohttp.client_proto.ResponseHandler object at 0x7b49302e5850>, 1301843.040276687)])']


       source                                              label         var time_coord  n_time               start                 end  median_dt_hours         resolution forecast_meta
        PRISM    1981-01-01_1989-12-31_daily_4km_PRISM_data.zarr         ppt       time  3287.0 1981-01-01 00:00:00 1989-12-31 00:00:00             24.0              daily          none
        PRISM    1990-01-01_1999-12-31_daily_4km_PRISM_data.zarr         ppt       time  3652.0 1990-01-01 00:00:00 1999-12-31 00:00:00             24.0              daily          none
        PRISM    2000-01-01_2009-12-31_daily_4km_PRISM_data.zarr         ppt       time  3653.0 2000-01-01 00:00:00 2009-12-31 00:00:00             24.0              daily          none
        PRISM    2010-01-01_2013-12-31_daily_4km_PRISM_data.zarr         ppt       time  1461.0 2010-01-01 00:00:00 2013-12-31 00:00:00             24.0              daily          none
        PRISM    2014-01-01_2014-12-31_daily_4km_PRISM_data.zarr      

Future exception was never retrieved
future: <Future finished exception=ClientConnectionError('Connection lost: SSL shutdown timed out')>
TimeoutError: SSL shutdown timed out

The above exception was the direct cause of the following exception:

aiohttp.client_exceptions.ClientConnectionError: Connection lost: SSL shutdown timed out
Future exception was never retrieved
future: <Future finished exception=ClientConnectionError('Connection lost: SSL shutdown timed out')>
TimeoutError: SSL shutdown timed out

The above exception was the direct cause of the following exception:

aiohttp.client_exceptions.ClientConnectionError: Connection lost: SSL shutdown timed out
Future exception was never retrieved
future: <Future finished exception=ClientConnectionError('Connection lost: SSL shutdown timed out')>
TimeoutError: SSL shutdown timed out

The above exception was the direct cause of the following exception:

aiohttp.client_exceptions.ClientConnectionError: Connection lost: SSL shutdown timed